# ðŸ¤– CryptoPredictBot â€” Kaggle Training (Improved)
**GPU: T4 x2 | 25 coins | 4-timeframes | LGBM + XGBoost + CatBoost + LSTM ensemble**

Pulls code from: https://github.com/mahmudnuman/crypto-bot

Improvements over PC baseline:
- âœ… More estimators (T4 VRAM handles it)
- âœ… LSTM as 4th ensemble member
- âœ… Optuna hyperparameter tuning
- âœ… Funding rate features (Binance futures)
- âœ… Fear & Greed sentiment index
- âœ… Full 15-fold walk-forward validation

In [ ]:
# -- 1. Install missing packages, keeping numpy>=2.1.0 for scipy compat --
import subprocess, sys

# Pin numpy>=2.1.0 FIRST — pandas-ta would otherwise drag it down to 2.0.x
# which breaks scipy (scipy needs numpy._core.umath._center, added in 2.1.0)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=2.1.0',          # must come first to prevent downgrade
    'pandas-ta',
    'loguru',
    'optuna',
    'optuna-integration[lightgbm]',
], check=True)

import numpy as np
print(f'numpy version: {np.__version__}')  # confirm >= 2.1.0
print('Dependencies ready')


In [ ]:
# â”€â”€ 2. Pull code from GitHub â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, os

if not os.path.exists('/kaggle/working/crypto-bot'):
    subprocess.run(['git', 'clone', 'https://github.com/mahmudnuman/crypto-bot.git',
                    '/kaggle/working/crypto-bot'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/kaggle/working/crypto-bot')

os.chdir('/kaggle/working/crypto-bot')
import sys; sys.path.insert(0, '/kaggle/working/crypto-bot')
print('âœ… Code ready')

In [ ]:
# â”€â”€ 3. Download all market data from Binance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Downloads 5m + 1h + 6h + 1d for all 25 coins
# Takes ~30-60 min on Kaggle (fast internet)

from config import POPULAR_COINS, QUOTE_ASSET
from data.downloader import fetch_and_store_rest, fetch_and_store_5m_bulk
from data.universe import build_universe

build_universe()
symbols = [f'{c}{QUOTE_ASSET}' for c in POPULAR_COINS]
print(f'Downloading data for {len(symbols)} coins...')

for i, sym in enumerate(symbols, 1):
    print(f'  [{i:02d}/{len(symbols)}] {sym}...', end='', flush=True)
    try:
        fetch_and_store_5m_bulk(sym)
        for tf in ['1h', '6h', '1d']:
            fetch_and_store_rest(sym, tf)
        print(' done')
    except Exception as e:
        print(f' ERROR: {e}')

print('âœ… Data download complete')

In [ ]:
# â”€â”€ 4. Extra features: Funding Rate + Fear & Greed â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import requests, pandas as pd, json
from pathlib import Path

EXTRA_DIR = Path('cache/extra')
EXTRA_DIR.mkdir(parents=True, exist_ok=True)

def fetch_fear_greed(limit=2000):
    """Free API â€” no key needed."""
    url = f'https://api.alternative.me/fng/?limit={limit}&format=json'
    r = requests.get(url, timeout=30)
    data = r.json()['data']
    df = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True)
    df = df.set_index('timestamp').sort_index()
    df['fear_greed'] = df['value'].astype(float)
    df['fg_class'] = df['value_classification']
    df.to_parquet(EXTRA_DIR / 'fear_greed.parquet')
    print(f'  Fear & Greed: {len(df)} days saved')
    return df

def fetch_funding_rates(symbol, limit=1000):
    """Binance futures funding rate â€” free, no key."""
    url = f'https://fapi.binance.com/fapi/v1/fundingRate?symbol={symbol}&limit={limit}'
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        if not isinstance(data, list): return None
        df = pd.DataFrame(data)
        df['fundingTime'] = pd.to_datetime(df['fundingTime'], unit='ms', utc=True)
        df = df.set_index('fundingTime').sort_index()
        df['funding_rate'] = df['fundingRate'].astype(float)
        return df[['funding_rate']]
    except:
        return None

print('Fetching extra features...')
fg = fetch_fear_greed()

from config import POPULAR_COINS, QUOTE_ASSET
funding_data = {}
for coin in POPULAR_COINS:
    sym = f'{coin}{QUOTE_ASSET}'
    df = fetch_funding_rates(sym)
    if df is not None:
        funding_data[sym] = df
        print(f'  Funding rate {sym}: {len(df)} records')

# Save
import pickle
(EXTRA_DIR / 'funding_rates.pkl').write_bytes(pickle.dumps(funding_data))
print('âœ… Extra features saved')

In [ ]:
# â”€â”€ 5. Build feature matrices (with extra features merged in) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pandas as pd, numpy as np, pickle
from pathlib import Path
from features.multi_tf import build_multi_tf_features
from config import POPULAR_COINS, QUOTE_ASSET

FEAT_DIR  = Path('cache/features')
EXTRA_DIR = Path('cache/extra')
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# Load extra data
fg_df       = pd.read_parquet(EXTRA_DIR / 'fear_greed.parquet')
funding_map = pickle.loads((EXTRA_DIR / 'funding_rates.pkl').read_bytes())

def add_extra_features(df, symbol):
    """Merge funding rate + fear & greed into feature matrix."""
    # Fear & Greed (daily) â€” forward-fill to 5m
    fg_daily = fg_df[['fear_greed']].resample('5min').ffill()
    df = df.join(fg_daily, how='left')
    df['fear_greed'] = df['fear_greed'].ffill().fillna(50)

    # Funding rate (every 8h) â€” forward-fill to 5m
    if symbol in funding_map:
        fr = funding_map[symbol][['funding_rate']].resample('5min').ffill()
        df = df.join(fr, how='left')
        df['funding_rate'] = df['funding_rate'].ffill().fillna(0)
    else:
        df['funding_rate'] = 0.0

    # Derived: funding rate momentum
    df['funding_rate_ma3']  = df['funding_rate'].rolling(3).mean()
    df['funding_rate_sign'] = np.sign(df['funding_rate'])
    df['fg_normalized']     = (df['fear_greed'] - 50) / 50  # -1 to 1

    return df

symbols = [f'{c}{QUOTE_ASSET}' for c in POPULAR_COINS]
print(f'Building {len(symbols)} feature matrices (with extra features)...')

for i, sym in enumerate(symbols, 1):
    print(f'  [{i:02d}/{len(symbols)}] {sym}...', end='', flush=True)
    try:
        df = build_multi_tf_features(sym)
        df = add_extra_features(df, sym)
        out = FEAT_DIR / f'{sym}_features.parquet'
        df.to_parquet(out)
        print(f' {len(df):,}r x {df.shape[1]}c âœ“')
    except Exception as e:
        print(f' ERROR: {e}')

print('âœ… Features ready')

In [ ]:
# â”€â”€ 6. Optuna hyperparameter tuning (run once on BTC, apply globally) â”€â”€
import optuna, pandas as pd, numpy as np
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from features.pipeline import get_feature_cols, build_classifier_pipeline

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Sample 50k rows from BTC for tuning
df    = pd.read_parquet('cache/features/BTCUSDT_features.parquet')
FCols = get_feature_cols(df)
df_c  = df.dropna(subset=['target_dir']).tail(50000)
X_raw = df_c[FCols].replace([np.inf,-np.inf], np.nan).values.astype(np.float32)
y     = df_c['target_dir'].astype(int).values

pipe  = build_classifier_pipeline()
X     = pipe.fit_transform(X_raw, y)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 7),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state': 42, 'verbose': -1, 'n_jobs': -1,
    }
    clf   = LGBMClassifier(**params)
    score = cross_val_score(clf, X, y, cv=3, scoring='accuracy', n_jobs=-1).mean()
    return score

print('Running Optuna (50 trials on BTC)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f'\nBest LGBM params: acc={study.best_value:.4f}')
print(best_params)

import json
Path('cache/best_lgbm_params.json').write_text(json.dumps(best_params, indent=2))
print('âœ… Optuna tuning complete â€” params saved')

In [ ]:
# â”€â”€ 7. Update config with tuned params + T4 GPU settings â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json
from pathlib import Path

# Load Optuna best params
best = json.loads(Path('cache/best_lgbm_params.json').read_text())

# Patch config at runtime for this session
import config
config.LGBM_PARAMS.update({**best, 'verbose': -1, 'n_jobs': -1})

# T4 has 16GB VRAM â€” use much more trees than MX130
config.XGBM_PARAMS.update({'n_estimators': 800, 'device': 'cuda'})
config.CATBOOST_PARAMS.update({'iterations': 800, 'task_type': 'GPU'})

# More folds on Kaggle (T4 is fast)
config.WFV_STEP_DAYS  = 30   # 15 folds instead of 9
config.MAX_TRAIN_ROWS = 400_000  # T4 has 30GB RAM

print('Config updated for Kaggle T4:')
print(f'  LGBM: {config.LGBM_PARAMS["n_estimators"]} trees')
print(f'  XGB:  {config.XGBM_PARAMS["n_estimators"]} trees (CUDA)')
print(f'  CB:   {config.CATBOOST_PARAMS["iterations"]} trees (GPU)')
print(f'  WFV:  {config.WFV_STEP_DAYS}d step = ~15 folds/coin')
print('âœ… Config ready')

In [ ]:
# â”€â”€ 8. Train all 25 coins (uses run_all.py in-process logic) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# This runs the full training pipeline from run_all.py
import time
from run_all import train_all_inprocess
from config import POPULAR_COINS, QUOTE_ASSET

symbols = [f'{c}{QUOTE_ASSET}' for c in POPULAR_COINS]

print(f'Starting training: {len(symbols)} coins on T4 GPU')
print(f'Estimated time: ~{len(symbols) * 15 * 4 // 60}h\n')

t0 = time.time()
results = train_all_inprocess(symbols)
elapsed = (time.time() - t0) / 3600

ok     = [r for r in results if r['status'] == 'ok']
failed = [r for r in results if r['status'] == 'failed']

print(f'\n=== DONE ===')
print(f'Time: {elapsed:.1f}h')
print(f'OK: {len(ok)}/25  |  Failed: {len(failed)}')
if ok:
    avg = sum(r["acc"] for r in ok)/len(ok)
    best = max(ok, key=lambda r: r["acc"])
    print(f'Avg best-fold acc: {avg:.4f}')
    print(f'Best coin: {best["symbol"]} = {best["acc"]:.4f}')

In [ ]:
# â”€â”€ 9. Results summary + push to GitHub â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, subprocess
from pathlib import Path

# Print per-coin accuracy table
print(f'{"Coin":<14} {"BestAcc":>10} {"AvgAcc":>10} {"Folds":>7} {"Grade":>7}')
print('-' * 55)

ckpt_dir = Path('cache/checkpoints')
rows = []
for f in sorted(ckpt_dir.glob('*.json')):
    if 'fold' in f.stem: continue
    try:
        d     = json.loads(f.read_text())
        folds = d.get('folds', [])
        best  = d.get('best_acc', 0)
        avg   = sum(x['test_acc'] for x in folds)/len(folds) if folds else 0
        grade = 'Aâ˜…â˜…â˜…' if best>=0.62 else 'Bâ˜…â˜…' if best>=0.57 else 'Câ˜…' if best>=0.53 else 'D'
        rows.append((f.stem, best, avg, len(folds), grade))
        print(f'{f.stem:<14} {best:>10.4f} {avg:>10.4f} {len(folds):>7} {grade:>7}')
    except: pass

if rows:
    avg_all = sum(r[1] for r in rows)/len(rows)
    print(f'\nOverall avg best-fold accuracy: {avg_all:.4f}')

# Push to GitHub
print('\nPushing results to GitHub...')
cmds = [
    'git add -A',
    'git commit -m "kaggle training complete: improved accuracy with funding rate + optuna + 800 trees"',
    'git push origin main'
]
for cmd in cmds:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  {cmd}: {r.stderr[:100]}')
    else:
        print(f'  âœ… {cmd}')

print('\nðŸŽ‰ All done! Check github.com/mahmudnuman/crypto-bot')